# Assignment 01 — Mapping the global submarine cable network

**Research question:** What does the geography and age of the global submarine cable
network reveal about the spatial concentration and continuing expansion of digital
infrastructure?

This notebook maps submarine cable routes worldwide and explores two attributes in
particular: the **ready-for-service year** (`rfs_year`) and whether a cable is
**planned or already in service** (`is_planned`). The analysis uses the public API of
TeleGeography's Submarine Cable Map, linked from the Copernicus Marine product
`EXT_TELEGEOGRAPHY_SUBMARINE_CABLES`.

Sources:

- Copernicus Marine product description: https://data.marine.copernicus.eu/product/EXT_TELEGEOGRAPHY_SUBMARINE_CABLES/description
- Cable geometries: https://www.submarinecablemap.com/api/v3/cable/cable-geo.json
- Cable metadata index: https://www.submarinecablemap.com/api/v3/cable/all.json


## 1. Setup

### Import the required libraries

This first code block loads every library used later in the notebook. The Python
standard-library modules manage paths, JSON data, text parsing, timing, and
concurrent downloads. GeoPandas and Shapely handle spatial data; pandas and NumPy
support tabular and numerical operations; Matplotlib creates the visualizations;
Requests retrieves data from the web APIs; and xarray opens the oceanographic
NetCDF datasets.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import time
from pathlib import Path

import geopandas as gpd
import mapbox_vector_tile
import matplotlib.pyplot as plt
import mercantile
import numpy as np
import pandas as pd
import requests
import xarray as xr
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.path import Path as MarkerPath
from shapely import contains_xy
from shapely.geometry import shape

### Configure paths, typography, and data sources

The next block prepares the working environment. It increases the number of pandas
columns shown in notebook outputs, locates the assignment folder, and creates the
local data directory when necessary. It also registers the Cascadia Code font so
all plots share consistent typography. Finally, it stores the API endpoints and
local filenames as constants, making later cells easier to read and update.


In [ ]:
pd.set_option("display.max_columns", 30)

PROJECT_ASSIGNMENTS = Path("cdp-mapping-systems/content/Assignments")
if PROJECT_ASSIGNMENTS.exists():
    ASSIGNMENTS_DIR = PROJECT_ASSIGNMENTS
elif (Path("..") / "assets").exists():
    ASSIGNMENTS_DIR = Path("..")
else:
    ASSIGNMENTS_DIR = Path(".")
DATA_DIR = ASSIGNMENTS_DIR / "01-Data/marine-cables"
DATA_DIR.mkdir(parents=True, exist_ok=True)

FONT_FILE = ASSIGNMENTS_DIR / "assets/fonts/CascadiaCode/CascadiaCode.ttf"
font_manager.fontManager.addfont(FONT_FILE)
FONT_FAMILY = font_manager.FontProperties(fname=FONT_FILE).get_name()
plt.rcParams["font.family"] = FONT_FAMILY

GEOMETRY_URL = "https://www.submarinecablemap.com/api/v3/cable/cable-geo.json"
LANDING_POINTS_URL = (
    "https://www.submarinecablemap.com/api/v3/landing-point/landing-point-geo.json"
)
INDEX_URL = "https://www.submarinecablemap.com/api/v3/cable/all.json"
DETAIL_URL = "https://www.submarinecablemap.com/api/v3/cable/{cable_id}.json"

GEOMETRY_FILE = DATA_DIR / "cable-geo.json"
LANDING_POINTS_FILE = DATA_DIR / "landing-point-geo.json"
METADATA_FILE = DATA_DIR / "cable-metadata.json"
OCEAN_FILE = DATA_DIR / "copernicus_surface_biogeochemistry_2026-08-05.nc"
BATHYMETRY_FILE = DATA_DIR / "gmrt_global_topography.nc"
EBSA_FILE = ASSIGNMENTS_DIR / "01-Data/ebsa/ebsa_repository.geojson"

## 2. Download and cache the route geometries

The API is a changing live source. To make the analysis reproducible within this
project, the downloaded response is saved locally. Deleting the cached file and
rerunning the cell will retrieve a newer snapshot.

After loading the TeleGeography GeoJSON, the code displays the dataset structure:
its dimensions, column names, data types, coordinate reference system, and geometry
types. The final command shows the first 10 rows so that we can inspect the cable
identifiers, names, colors, representative coordinates, and route geometries.


In [ ]:
def download_if_missing(url, destination):
    if destination.exists():
        print(f"Using cached file: {destination}")
        return

    response = requests.get(url, timeout=60)
    response.raise_for_status()
    destination.write_bytes(response.content)
    print(f"Downloaded: {destination} ({destination.stat().st_size:,} bytes)")


download_if_missing(GEOMETRY_URL, GEOMETRY_FILE)
cables = gpd.read_file(GEOMETRY_FILE)
cables = cables.set_crs("EPSG:4326", allow_override=True)

print("TELEGEOGRAPHY DATASET STRUCTURE")
print(f"Rows: {cables.shape[0]:,}")
print(f"Columns: {cables.shape[1]}")
print(f"Column names: {cables.columns.tolist()}")
print(f"CRS: {cables.crs}")
print(f"Geometry types: {cables.geometry.geom_type.value_counts().to_dict()}")

display(cables.dtypes.rename("data_type").to_frame())
display(cables.head(10))

### Initial attribute inspection

The geometry endpoint contains route shapes plus a small set of descriptive fields.
`id` identifies the cable system, while `feature_id` distinguishes geometry features.
Multiple geometry features may therefore refer to the same cable system. This means
the number of rows should not automatically be interpreted as the number of cables.


In [ ]:
scalar_attributes = cables.drop(columns=["geometry", "coordinates"], errors="ignore")
attribute_summary = pd.DataFrame(
    {
        "dtype": scalar_attributes.dtypes.astype(str),
        "missing": scalar_attributes.isna().sum(),
        "unique": scalar_attributes.nunique(dropna=True),
    }
)
attribute_summary

In [ ]:
print(f"Geometry features: {len(cables):,}")
print(f"Unique cable IDs: {cables['id'].nunique():,}")
print("Spatial bounds [west, south, east, north]:")
print(cables.total_bounds)

## 3. Global map with surface chlorophyll

Each line represents a generalized cable route supplied by TeleGeography. The map
reveals the spatial structure of the network: dense corridors connect major coastal
regions, while some ocean areas have comparatively few routes. The background layer is
Copernicus Marine total chlorophyll (`chl`) for 5 August 2026 at approximately 0.49 m
depth. Five fuchsia pattern densities represent relative chlorophyll quantiles, from
very low to very high concentrations, without using a continuous color gradient. Thin
grey bathymetric contours derived from the GMRT global topography grid show seafloor
depths at -8000, -6000, -4000, -2000, -1000, -500, -200, and -100 metres.

Line overlap is meaningful as an indication of mapped route concentration, but it is
not a direct measurement of bandwidth, traffic, redundancy, or economic importance. The
chlorophyll field is a single-day model product and should not be interpreted as a
long-term ecological average or as a causal explanation for cable locations.


In [ ]:
ocean = xr.open_dataset(OCEAN_FILE)
chlorophyll = (
    ocean["chl"]
    .isel(time=0, depth=0)
    .where(ocean["chl"].isel(time=0, depth=0) > 0)
    .load()
)
chlorophyll_date = pd.to_datetime(ocean["time"].values[0]).strftime("%Y-%m-%d")
chlorophyll_depth = float(ocean["depth"].values[0])
pattern_levels = [
    float(chlorophyll.quantile(q, skipna=True)) for q in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
]
pattern_labels = ["Very low", "Low", "Medium", "High", "Very high"]
pattern_strides = [12, 9, 7, 5, 3]  # smaller stride = closer marks
slash_marker = MarkerPath(
    [[-0.5, -0.5], [0.5, 0.5]], [MarkerPath.MOVETO, MarkerPath.LINETO]
)

ebsa = gpd.read_file(EBSA_FILE).set_crs("OGC:CRS84", allow_override=True)

# Match every route to the planned/in-service status supplied by TeleGeography
map_metadata = pd.DataFrame(
    json.loads(METADATA_FILE.read_text(encoding="utf-8"))["records"]
)[["id", "is_planned"]]
cables_by_status = cables.merge(map_metadata, on="id", how="left")
in_service_cables = cables_by_status[~cables_by_status["is_planned"].fillna(False)]
planned_cables = cables_by_status[cables_by_status["is_planned"].eq(True)]

gmrt = xr.open_dataset(BATHYMETRY_FILE)
gmrt_width, gmrt_height = [int(value) for value in gmrt["dimension"].values]
gmrt_longitude = np.linspace(
    gmrt["x_range"].values[0], gmrt["x_range"].values[1], gmrt_width
)
gmrt_latitude = np.linspace(
    gmrt["y_range"].values[1], gmrt["y_range"].values[0], gmrt_height
)
gmrt_elevation = gmrt["z"].values.reshape(gmrt_height, gmrt_width)
gmrt_bathymetry = np.where(gmrt_elevation < 0, gmrt_elevation, np.nan)
bathymetry_levels = [-8000, -6000, -4000, -2000, -1000, -500, -200, -100]

fig, ax = plt.subplots(figsize=(18, 9), facecolor="white")
ax.set_facecolor("white")

longitude_grid, latitude_grid = np.meshgrid(
    ocean["longitude"].values, ocean["latitude"].values
)
chlorophyll_values = chlorophyll.values
for class_index, stride in enumerate(pattern_strides):
    sample = (slice(None, None, stride), slice(None, None, stride))
    sampled_values = chlorophyll_values[sample]
    lower = pattern_levels[class_index]
    upper = pattern_levels[class_index + 1]
    if class_index == len(pattern_strides) - 1:
        class_mask = (sampled_values >= lower) & (sampled_values <= upper)
    else:
        class_mask = (sampled_values >= lower) & (sampled_values < upper)
    class_mask &= np.isfinite(sampled_values)
    ax.scatter(
        longitude_grid[sample][class_mask],
        latitude_grid[sample][class_mask],
        marker=slash_marker,
        s=7,
        facecolors="none",
        edgecolors="#FF00D4",
        linewidths=0.28,
        alpha=0.72,
        zorder=0,
    )

# GMRT bathymetric contours
ax.contour(
    gmrt_longitude,
    gmrt_latitude,
    gmrt_bathymetry,
    levels=bathymetry_levels,
    colors="#5A5A5A",
    linewidths=0.34,
)

# CBD EBSAs: small cyan crosses, without polygon borders
ebsa_pattern_geometry = ebsa.geometry.make_valid().union_all()
ebsa_pattern_longitudes = np.arange(-179.25, 180, 1.5)
ebsa_pattern_latitudes = np.arange(-69.25, 85, 1.5)
ebsa_pattern_x, ebsa_pattern_y = np.meshgrid(
    ebsa_pattern_longitudes, ebsa_pattern_latitudes
)
ebsa_pattern_mask = contains_xy(
    ebsa_pattern_geometry,
    ebsa_pattern_x.ravel(),
    ebsa_pattern_y.ravel(),
)
ax.scatter(
    ebsa_pattern_x.ravel()[ebsa_pattern_mask],
    ebsa_pattern_y.ravel()[ebsa_pattern_mask],
    marker="+",
    s=7,
    color="#00FFFB",
    linewidths=0.34,
    alpha=0.72,
    zorder=0.8,
)

# Existing cable routes in electric blue
in_service_cables.plot(
    ax=ax,
    color="#0015FF",
    linewidth=0.35,
    alpha=0.82,
    zorder=2,
)

# Planned future cable routes: violet glow and crisp foreground
planned_cables.plot(
    ax=ax,
    color="#C45CFF",
    linewidth=2.2,
    alpha=0.24,
    zorder=2.8,
)
planned_cables.plot(
    ax=ax,
    color="#8300FF",
    linewidth=0.35,
    linestyle="--",
    alpha=1.0,
    zorder=3,
)

ax.set_xlim(-180, 180)
ax.set_ylim(-70, 85)
ax.set_title(
    f"Global submarine cable routes, EBSAs and surface chlorophyll - {chlorophyll_date}",
    color="#111111",
    fontsize=18,
    pad=14,
)
ebsa_handle = Line2D(
    [0],
    [0],
    linestyle="none",
    marker="+",
    markersize=6,
    markeredgewidth=0.45,
    color="#00DFF2",
    label="CBD EBSA",
)
cable_handles = [
    Line2D([0], [0], color="#0015FF", linewidth=1, label="In service cables"),
    Line2D(
        [0], [0], color="#8300FF", linewidth=1, linestyle="--", label="Planned cables"
    ),
]
chlorophyll_handle = Line2D(
    [0],
    [0],
    linestyle="none",
    marker=slash_marker,
    markersize=7,
    markeredgewidth=0.4,
    markeredgecolor="#FF00D4",
    label="Surface chlorophyll",
)
legend = ax.legend(
    handles=[*cable_handles, ebsa_handle, chlorophyll_handle],
    title="Map layers",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.105),
    ncol=4,
    frameon=False,
)
plt.setp(legend.get_texts(), color="#222222")
legend.get_title().set_color("#111111")
fig.text(
    0.01,
    0.01,
    "Sources: TeleGeography; E.U. Copernicus Marine Service (CMEMS); GMRT v4.5; CBD EBSA Repository",
    color="#666666",
    fontsize=9,
)
ax.set_axis_off()
plt.tight_layout(rect=[0, 0.075, 1, 1])
ocean.close()
gmrt.close()
plt.show()

print(
    f"Rendered cable routes — in service: {len(in_service_cables):,}; "
    f"planned in violet: {len(planned_cables):,}"
)

## 4. Map with cable colour intensity by ready-for-service year

This cell keeps the previous map layout (chlorophyll pattern, bathymetry, EBSAs) and
only changes how cable routes are drawn. Colour intensity now follows `rfs_year`: older
systems appear lighter, newer systems appear more saturated. In-service routes stay on
a blue scale; planned routes stay on a violet dashed scale. Routes without an RFS year
are drawn in grey.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

ocean = xr.open_dataset(OCEAN_FILE)
chlorophyll = (
    ocean["chl"]
    .isel(time=0, depth=0)
    .where(ocean["chl"].isel(time=0, depth=0) > 0)
    .load()
)
chlorophyll_date = pd.to_datetime(ocean["time"].values[0]).strftime("%Y-%m-%d")
chlorophyll_depth = float(ocean["depth"].values[0])
pattern_levels = [
    float(chlorophyll.quantile(q, skipna=True)) for q in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
]
pattern_labels = ["Very low", "Low", "Medium", "High", "Very high"]
pattern_strides = [12, 9, 7, 5, 3]  # smaller stride = closer marks
slash_marker = MarkerPath(
    [[-0.5, -0.5], [0.5, 0.5]], [MarkerPath.MOVETO, MarkerPath.LINETO]
)

ebsa = gpd.read_file(EBSA_FILE).set_crs("OGC:CRS84", allow_override=True)

# Match every route to status and ready-for-service year
map_metadata = pd.DataFrame(
    json.loads(METADATA_FILE.read_text(encoding="utf-8"))["records"]
)[["id", "is_planned", "rfs_year"]]
map_metadata["rfs_year"] = pd.to_numeric(map_metadata["rfs_year"], errors="coerce")
cables_by_year = cables.merge(map_metadata, on="id", how="left")
in_service_by_year = cables_by_year[~cables_by_year["is_planned"].fillna(False)].copy()
planned_by_year = cables_by_year[cables_by_year["is_planned"].eq(True)].copy()

year_min = float(cables_by_year["rfs_year"].min())
year_max = float(cables_by_year["rfs_year"].max())
year_norm = Normalize(vmin=year_min, vmax=year_max)
blue_by_year = LinearSegmentedColormap.from_list(
    "cable_blue_by_year", ["#2B45C3", "#D4D4FF"]
)
violet_by_year = LinearSegmentedColormap.from_list(
    "cable_violet_by_year", ["#F700FF", "#F700FF"]
)

gmrt = xr.open_dataset(BATHYMETRY_FILE)
gmrt_width, gmrt_height = [int(value) for value in gmrt["dimension"].values]
gmrt_longitude = np.linspace(
    gmrt["x_range"].values[0], gmrt["x_range"].values[1], gmrt_width
)
gmrt_latitude = np.linspace(
    gmrt["y_range"].values[1], gmrt["y_range"].values[0], gmrt_height
)
gmrt_elevation = gmrt["z"].values.reshape(gmrt_height, gmrt_width)
gmrt_bathymetry = np.where(gmrt_elevation < 0, gmrt_elevation, np.nan)
bathymetry_levels = [
    -8000,
    -7000,
    -6000,
    -5000,
    -4000,
    -3000,
    -2000,
    -1000,
    -900,
    -800,
    -700,
    -600,
    -500,
    -400,
    -300,
    -200,
    -100,
]

fig, ax = plt.subplots(figsize=(30, 16), dpi=600, facecolor="#090A18")
ax.set_facecolor("#090A18")

longitude_grid, latitude_grid = np.meshgrid(
    ocean["longitude"].values, ocean["latitude"].values
)
chlorophyll_values = chlorophyll.values
for class_index, stride in enumerate(pattern_strides):
    sample = (slice(None, None, stride), slice(None, None, stride))
    sampled_values = chlorophyll_values[sample]
    lower = pattern_levels[class_index]
    upper = pattern_levels[class_index + 1]
    if class_index == len(pattern_strides) - 1:
        class_mask = (sampled_values >= lower) & (sampled_values <= upper)
    else:
        class_mask = (sampled_values >= lower) & (sampled_values < upper)
    class_mask &= np.isfinite(sampled_values)
    ax.scatter(
        longitude_grid[sample][class_mask],
        latitude_grid[sample][class_mask],
        marker=slash_marker,
        s=7,
        facecolors="none",
        edgecolors="#FF00D4",
        linewidths=0.28,
        alpha=0,
        zorder=0,
    )

# GMRT bathymetric contours
ax.contour(
    gmrt_longitude,
    gmrt_latitude,
    gmrt_bathymetry,
    levels=bathymetry_levels,
    colors="#FFC8F7",
    linewidths=0.25,
)

# In-service cables: blue intensity scales with RFS year
in_service_by_year.plot(
    ax=ax,
    column="rfs_year",
    cmap=blue_by_year,
    norm=year_norm,
    linewidth=1,
    alpha=0.88,
    zorder=2,
    legend=False,
    missing_kwds={"color": "#9AA0A6", "linewidth": 0.28, "alpha": 0.45},
)

# Planned cables: violet intensity scales with RFS year
planned_by_year.plot(
    ax=ax,
    column="rfs_year",
    cmap=violet_by_year,
    norm=year_norm,
    linewidth=1,
    alpha=0.22,
    zorder=2.8,
    legend=False,
    missing_kwds={"color": "#C9B0D9", "linewidth": 1.6, "alpha": 0.18},
)
planned_by_year.plot(
    ax=ax,
    column="rfs_year",
    cmap=violet_by_year,
    norm=year_norm,
    linewidth=0.35,
    linestyle="--",
    alpha=1.0,
    zorder=3,
    legend=False,
    missing_kwds={
        "color": "#9AA0A6",
        "linewidth": 0.28,
        "linestyle": "--",
        "alpha": 0.55,
    },
)

ax.set_xlim(-180, 180)
ax.set_ylim(-70, 85)
ax.set_title(
    f"Cable routes by RFS year intensity, EBSAs and surface chlorophyll - {chlorophyll_date}",
    color="#111111",
    fontsize=18,
    pad=14,
)
ebsa_handle = Line2D(
    [0],
    [0],
    linestyle="none",
    marker="+",
    markersize=6,
    markeredgewidth=0.45,
    color="#00DFF2",
    label="CBD EBSA",
)
cable_handles = [
    Line2D(
        [0], [0], color="#0015FF", linewidth=1.4, label="In service (darker = newer)"
    ),
    Line2D(
        [0],
        [0],
        color="#FF44E0",
        linewidth=1.4,
        linestyle="--",
        label="Planned (darker = newer)",
    ),
]
chlorophyll_handle = Line2D(
    [0],
    [0],
    linestyle="none",
    marker=slash_marker,
    markersize=7,
    markeredgewidth=0.4,
    markeredgecolor="#FF00D4",
    label="Surface chlorophyll",
)

plt.setp(legend.get_texts(), color="#222222")
legend.get_title().set_color("#111111")

# Colour bar for ready-for-service year
sm = ScalarMappable(norm=year_norm, cmap=blue_by_year)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    fraction=0.035,
    pad=0.02,
    aspect=42,
)
cbar.set_label("Ready-for-service year (colour intensity)", color="#222222")
cbar.ax.xaxis.set_tick_params(color="#444444")
plt.setp(cbar.ax.get_xticklabels(), color="#333333")

fig.text(
    0.01,
    0.01,
    "Sources: TeleGeography; E.U. Copernicus Marine Service (CMEMS); GMRT v4.5; CBD EBSA Repository",
    color="#666666",
    fontsize=9,
)
ax.set_axis_off()
plt.tight_layout(rect=[0, 0.075, 1, 1])
ocean.close()
gmrt.close()
plt.show()

print(
    f"Rendered year-scaled routes — in service: {len(in_service_by_year):,}; "
    f"planned: {len(planned_by_year):,}; "
    f"RFS year range: {int(year_min)}-{int(year_max)}"
)

## 5. Map coloured by ownership category

This copy keeps the previous dark basemap, bathymetry, and landing points, but replaces
the random neon assignment with the ownership classification:

- **Telco / consortium only** — no Google, Meta, Microsoft, or Amazon/AWS in `owners`
- **Google / Meta / Microsoft / Amazon** — exactly one of these big-tech names appears
  (including hybrids with traditional operators)
- **Multiple big tech** — two or more of those firms appear together
- Planned routes remain dashed

The same neon palette is reused so the visual language stays consistent with map 3B.


In [ ]:
# Ownership-coloured copy of the 3B landing-point map (same palette, meaningful colours)
download_if_missing(LANDING_POINTS_URL, LANDING_POINTS_FILE)
landing_points_own = gpd.read_file(LANDING_POINTS_FILE).to_crs("EPSG:4326")

neon_palette = [
    "#F15BB5",  # magenta  -> telco / consortium
    "#00CFE8",  # cyan  -> Microsoft
    "#1C8CFF",  # blue  -> Google
    "#9B5DE5",  # violet -> multiple big tech
    "#FEE440",  # pink  -> Meta
    "#56E0C5",  # mint (unused spare)
    "#FEE440",  # yellow -> landing points (kept)
    "#FF8C42",  # orange -> Amazon / AWS
]

BIG_TECH_LABELS = {
    "Google": "Google",
    "Meta": "Meta",
    "Facebook": "Meta",
    "Microsoft": "Microsoft",
    "Amazon": "Amazon",
    "Amazon Web Services": "Amazon",
    "AWS": "Amazon",
}

ownership_colors = {
    "Telco / consortium": neon_palette[0],
    "Google": neon_palette[2],
    "Meta": neon_palette[4],
    "Microsoft": neon_palette[1],
    "Amazon": neon_palette[7],
    "Multiple big tech": neon_palette[3],
    "Unknown": "#6F7898",
}


def split_owners(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    text = str(raw).strip()
    if not text:
        return []
    return [part.strip() for part in text.split(",") if part.strip()]


def ownership_group(raw_owners):
    owners = split_owners(raw_owners)
    if not owners:
        return "Unknown"
    big = []
    for owner in owners:
        label = BIG_TECH_LABELS.get(owner)
        if label and label not in big:
            big.append(label)
    if not big:
        return "Telco / consortium"
    if len(big) == 1:
        return big[0]
    return "Multiple big tech"


ownership_meta = pd.DataFrame(
    json.loads(METADATA_FILE.read_text(encoding="utf-8"))["records"]
)[["id", "is_planned", "owners", "name"]]

owned_cables = cables.merge(ownership_meta, on="id", how="left", suffixes=("", "_meta"))
if (
    "is_planned" not in owned_cables.columns
    and "is_planned_meta" in owned_cables.columns
):
    owned_cables["is_planned"] = owned_cables["is_planned_meta"]
owned_cables["ownership_group"] = owned_cables["owners"].apply(ownership_group)
owned_cables["cable_color"] = owned_cables["ownership_group"].map(ownership_colors)

owned_in_service = owned_cables[~owned_cables["is_planned"].fillna(False)]
owned_planned = owned_cables[owned_cables["is_planned"].eq(True)]

gmrt_own = xr.open_dataset(BATHYMETRY_FILE)
gmrt_w, gmrt_h = [int(value) for value in gmrt_own["dimension"].values]
gmrt_lon = np.linspace(
    gmrt_own["x_range"].values[0], gmrt_own["x_range"].values[1], gmrt_w
)
gmrt_lat = np.linspace(
    gmrt_own["y_range"].values[1], gmrt_own["y_range"].values[0], gmrt_h
)
gmrt_elev = gmrt_own["z"].values.reshape(gmrt_h, gmrt_w)
gmrt_bathy = np.where(gmrt_elev < 0, gmrt_elev, np.nan)
bathymetry_levels_own = [
    -8000,
    -7000,
    -6000,
    -5000,
    -4000,
    -3000,
    -2000,
    -1000,
    -900,
    -800,
    -700,
    -600,
    -500,
    -400,
    -300,
    -200,
    -100,
]

fig, ax = plt.subplots(figsize=(30, 16), dpi=180, facecolor="#090A18")
ax.set_facecolor("#090A18")
ax.contour(
    gmrt_lon,
    gmrt_lat,
    gmrt_bathy,
    levels=bathymetry_levels_own,
    colors="#30344F",
    linewidths=0.35,
    alpha=0.75,
    zorder=0,
)

# Draw telco routes first (context), then big-tech categories on top
draw_order = [
    "Telco / consortium",
    "Unknown",
    "Amazon",
    "Microsoft",
    "Meta",
    "Google",
    "Multiple big tech",
]
for group_name in draw_order:
    group_service = owned_in_service[owned_in_service["ownership_group"].eq(group_name)]
    group_planned = owned_planned[owned_planned["ownership_group"].eq(group_name)]
    if len(group_service):
        alpha = 0.35 if group_name in {"Telco / consortium", "Unknown"} else 0.92
        width = 1.2 if group_name in {"Telco / consortium", "Unknown"} else 0.70
        group_service.plot(
            ax=ax,
            color=ownership_colors[group_name],
            linewidth=width,
            alpha=alpha,
            zorder=2,
        )
    if len(group_planned):
        alpha = 0.45 if group_name in {"Telco / consortium", "Unknown"} else 0.95
        width = 1.2 if group_name in {"Telco / consortium", "Unknown"} else 0.90
        group_planned.plot(
            ax=ax,
            color=ownership_colors[group_name],
            linewidth=width,
            linestyle="--",
            alpha=alpha,
            zorder=2.5,
        )

ax.scatter(
    landing_points_own.geometry.x,
    landing_points_own.geometry.y,
    marker="+",
    s=11,
    color="#FFFFFF",
    linewidths=0.5,
    alpha=0.95,
    zorder=4,
)

ownership_handles = [
    Line2D([0], [0], color=ownership_colors[name], linewidth=2.2, label=name)
    for name in [
        "Telco / consortium",
        "Google",
        "Meta",
        "Microsoft",
        "Amazon",
        "Multiple big tech",
    ]
]
landing_handle_own = Line2D(
    [0],
    [0],
    linestyle="none",
    marker="+",
    markersize=7,
    markeredgewidth=0.8,
    color="#FFFFFF",
    label="Landing point",
)
planned_handle_own = Line2D(
    [0],
    [0],
    color="#B7F4E8",
    linewidth=1.2,
    linestyle="--",
    label="Planned route",
)
legend = ax.legend(
    handles=[*ownership_handles, landing_handle_own, planned_handle_own],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.075),
    ncol=8,
    frameon=False,
    fontsize=15,
)
plt.setp(legend.get_texts(), color="#FFFFFF")

ax.set_axis_off()
plt.tight_layout(rect=[0, 0.055, 1, 1])
gmrt_own.close()
plt.show()

counts = owned_cables.drop_duplicates("id")["ownership_group"].value_counts()
print("Cable systems by ownership group:")
print(counts.to_string())
print(f"Rendered {len(owned_cables):,} route features with ownership colours.")

## 6. Cable routes and WWF whale migration corridors

This third version preserves the previous maps and adds the migration-corridor
polygons published by the WWF Protecting Blue Corridors platform. Eight species are
included: blue, bowhead, fin, gray, humpback, North Atlantic right, southern right,
and sperm whales. The corridors are retrieved from the platform's official vector-tile
services at a global overview zoom level.

The pale pink fill and brighter pink outline form a contextual reading layer behind
the cable routes. They indicate broad migration areas rather than the movement of one
individual whale or a precise navigational path. Overlap with a cable is spatial
co-occurrence only and does not, by itself, demonstrate ecological impact.


In [ ]:
WWF_CORRIDOR_TILES = {
    "Sperm whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/sperm-whale-movement-fixgeo-landcut-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "North Atlantic right whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/NorthAtlanticRight-whale-movement-fixgeo-landcut-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "Fin whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/Fin-whale-movement-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "Humpback whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/humpback-whales-global-landcut-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "Southern right whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/SouthernRight-whale-movement-fixgeo-landcut-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "Gray whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/Gray-whale-movement-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "Bowhead whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/Bowhead-whale-movement-fixgeo-landcut-Jan2025/tiles/{z}/{x}/{y}.pbf",
    "Blue whale": "https://mbtileserver.bluecorridors.org/services/wwf_blue_corridors_corridors/Bluewhale-movement-fixgeo-landcut-Jan2025.cpg/tiles/{z}/{x}/{y}.pbf",
}


def vector_tile_transformer(tile_x, tile_y, zoom, extent=4096):
    bounds = mercantile.xy_bounds(tile_x, tile_y, zoom)

    def transform(x, y):
        mercator_x = bounds.left + (x / extent) * (bounds.right - bounds.left)
        mercator_y = bounds.bottom + (y / extent) * (bounds.top - bounds.bottom)
        lng_lat = mercantile.lnglat(mercator_x, mercator_y)
        return lng_lat.lng, lng_lat.lat

    return transform


def load_wwf_corridors(zoom=1):
    records = []
    for species, url_template in WWF_CORRIDOR_TILES.items():
        for tile in mercantile.tiles(-180, -85, 180, 85, zooms=zoom):
            response = requests.get(
                url_template.format(z=tile.z, x=tile.x, y=tile.y),
                timeout=60,
            )
            response.raise_for_status()
            decoded = mapbox_vector_tile.decode(
                response.content,
                default_options={
                    "transformer": vector_tile_transformer(tile.x, tile.y, tile.z),
                },
            )
            for layer in decoded.values():
                for feature in layer["features"]:
                    records.append(
                        {
                            "species": species,
                            "geometry": shape(feature["geometry"]),
                        }
                    )
    return gpd.GeoDataFrame(records, crs="EPSG:4326")


# Self-contained layers (do not depend on variables from map 5)
download_if_missing(LANDING_POINTS_URL, LANDING_POINTS_FILE)
landing_points = gpd.read_file(LANDING_POINTS_FILE).to_crs("EPSG:4326")

map6_meta = pd.DataFrame(
    json.loads(METADATA_FILE.read_text(encoding="utf-8"))["records"]
)[["id", "is_planned"]]
cables_map6 = cables.merge(map6_meta, on="id", how="left")
styled_in_service = cables_map6[~cables_map6["is_planned"].fillna(False)].copy()
styled_planned = cables_map6[cables_map6["is_planned"].eq(True)].copy()
styled_in_service["cable_color"] = "#56E0C5"
styled_planned["cable_color"] = "#B7F4E8"

gmrt_alt = xr.open_dataset(BATHYMETRY_FILE)
gmrt_w_alt, gmrt_h_alt = [int(value) for value in gmrt_alt["dimension"].values]
gmrt_longitude_alt = np.linspace(
    gmrt_alt["x_range"].values[0], gmrt_alt["x_range"].values[1], gmrt_w_alt
)
gmrt_latitude_alt = np.linspace(
    gmrt_alt["y_range"].values[1], gmrt_alt["y_range"].values[0], gmrt_h_alt
)
gmrt_elevation_alt = gmrt_alt["z"].values.reshape(gmrt_h_alt, gmrt_w_alt)
gmrt_bathymetry_alt = np.where(gmrt_elevation_alt < 0, gmrt_elevation_alt, np.nan)

wwf_corridors = load_wwf_corridors(zoom=1)

fig, ax = plt.subplots(figsize=(30, 16), dpi=120, facecolor="#090A18")
ax.set_facecolor("#090A18")
ax.contour(
    gmrt_longitude_alt,
    gmrt_latitude_alt,
    gmrt_bathymetry_alt,
    levels=[-8000, -6000, -4000, -2000, -1000, -500, -200, -100],
    colors="#30344F",
    linewidths=0.28,
    alpha=0.75,
    zorder=0,
)

wwf_corridors.plot(
    ax=ax,
    color="#F28CB1",
    edgecolor="#FFB3CB",
    linewidth=0.7,
    alpha=0.16,
    zorder=1,
)
styled_in_service.plot(
    ax=ax,
    color=styled_in_service["cable_color"],
    linewidth=0.48,
    alpha=0.30,
    zorder=2,
)
styled_planned.plot(
    ax=ax,
    color=styled_planned["cable_color"],
    linewidth=0.30,
    linestyle="--",
    alpha=0.95,
    zorder=2.5,
)
ax.scatter(
    landing_points.geometry.x,
    landing_points.geometry.y,
    marker="+",
    s=11,
    color="#FFFFFF",
    linewidths=0.25,
    alpha=0.95,
    zorder=4,
)
# Coordinate text labels omitted: ~2k labels freeze the notebook renderer

ax.set_xlim(-180, 180)
ax.set_ylim(-70, 85)
ax.set_title(
    "Submarine cable routes, landing points and WWF whale migration corridors",
    color="#56E0C5",
    fontsize=24,
    loc="left",
    pad=18,
)
corridor_handle = Line2D(
    [0], [0], color="#F28CB1", linewidth=7, alpha=0.55,
    label="WWF whale migration corridors",
)
landing_handle_wwf = Line2D(
    [0], [0], linestyle="none", marker="+", markersize=7,
    markeredgewidth=0.8, color="#FFFFFF", label="Landing point",
)
planned_handle_wwf = Line2D(
    [0], [0], color="#B7F4E8", linewidth=1.2, linestyle="--",
    label="Planned cable route",
)
legend = ax.legend(
    handles=[corridor_handle, landing_handle_wwf, planned_handle_wwf],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.075),
    ncol=3,
    frameon=False,
    fontsize=8,
)
plt.setp(legend.get_texts(), color="#B7F4E8")
fig.text(
    0.012,
    0.012,
    "Sources: TeleGeography; WWF Protecting Blue Corridors; GMRT v4.5",
    color="#6F7898",
    fontsize=8,
)
ax.set_axis_off()
plt.tight_layout(rect=[0, 0.055, 1, 1])
gmrt_alt.close()
plt.show()
plt.close(fig)

print(f"Rendered {len(wwf_corridors):,} WWF corridor polygon features.")
print(f"Species represented: {wwf_corridors['species'].nunique()}")


## 7. Public observed whale tracks

This fourth map keeps the cable routes, landing points, and WWF corridor polygons, then adds **observed satellite trajectories** as bright pink lines. The points come from the open supplementary dataset published with Hucke-Gaete et al. (2018), *From Chilean Patagonia to Galapagos, Ecuador* (DOI: 10.7717/peerj.4695/supp-1).

The dataset contains raw ARGOS-derived positions. We parse the timestamps, remove invalid or duplicate coordinate records, order observations chronologically for each tag, and connect consecutive observations belonging to the same tag. These tracks are a documented public subset from the eastern South Pacific; they are **not** the complete tracking collection behind the WWF platform. A zoom panel makes the observed trajectories readable without removing their global context.


In [ ]:
from matplotlib.patches import Patch
from shapely.geometry import LineString

TRACKS_FILE = (
    DATA_DIR / "PMC5933318_supplementary/peerj4695_blue_whale_argos_tracks.csv"
)
track_points = pd.read_csv(TRACKS_FILE)
track_points["date"] = pd.to_datetime(track_points["date"], errors="coerce")
track_points["lat"] = pd.to_numeric(track_points["lat"], errors="coerce")
track_points["long"] = pd.to_numeric(track_points["long"], errors="coerce")
track_points = (
    track_points.dropna(subset=["id", "date", "lat", "long"])
    .loc[lambda frame: frame["lat"].between(-90, 90) & frame["long"].between(-180, 180)]
    .drop_duplicates(subset=["id", "date", "lat", "long"])
    .sort_values(["id", "date"])
)

track_records = []
for tag_id, observations in track_points.groupby("id", sort=False):
    coordinates = list(zip(observations["long"], observations["lat"]))
    if len(coordinates) >= 2:
        track_records.append(
            {"tag_id": str(tag_id), "geometry": LineString(coordinates)}
        )
observed_tracks = gpd.GeoDataFrame(track_records, crs="EPSG:4326")

# Rebuild shared map layers locally so this cell does not depend on map 5/6 state
download_if_missing(LANDING_POINTS_URL, LANDING_POINTS_FILE)
landing_points = gpd.read_file(LANDING_POINTS_FILE).to_crs("EPSG:4326")

map7_meta = pd.DataFrame(
    json.loads(METADATA_FILE.read_text(encoding="utf-8"))["records"]
)[["id", "is_planned"]]
cables_map7 = cables.merge(map7_meta, on="id", how="left")
styled_in_service = cables_map7[~cables_map7["is_planned"].fillna(False)].copy()
styled_planned = cables_map7[cables_map7["is_planned"].eq(True)].copy()
styled_in_service["cable_color"] = "#56E0C5"
styled_planned["cable_color"] = "#B7F4E8"

gmrt_alt = xr.open_dataset(BATHYMETRY_FILE)
gmrt_w_alt, gmrt_h_alt = [int(value) for value in gmrt_alt["dimension"].values]
gmrt_longitude_alt = np.linspace(
    gmrt_alt["x_range"].values[0], gmrt_alt["x_range"].values[1], gmrt_w_alt
)
gmrt_latitude_alt = np.linspace(
    gmrt_alt["y_range"].values[1], gmrt_alt["y_range"].values[0], gmrt_h_alt
)
gmrt_elevation_alt = gmrt_alt["z"].values.reshape(gmrt_h_alt, gmrt_w_alt)
gmrt_bathymetry_alt = np.where(gmrt_elevation_alt < 0, gmrt_elevation_alt, np.nan)

if "wwf_corridors" not in globals() or wwf_corridors is None or len(wwf_corridors) == 0:
    raise RuntimeError("Run section 6 first so wwf_corridors is loaded.")


def draw_map_layers(target_ax):
    target_ax.set_facecolor("#090A18")
    target_ax.contour(
        gmrt_longitude_alt,
        gmrt_latitude_alt,
        gmrt_bathymetry_alt,
        levels=[-8000, -6000, -4000, -2000, -1000, -500, -200, -100],
        colors="#30344F",
        linewidths=0.28,
        alpha=0.75,
        zorder=0,
    )
    wwf_corridors.plot(
        ax=target_ax,
        facecolor="#F28CB1",
        edgecolor="#FFB3CB",
        linewidth=0.55,
        alpha=0.18,
        zorder=1.2,
    )
    styled_in_service.plot(
        ax=target_ax,
        color=styled_in_service["cable_color"],
        linewidth=0.45,
        alpha=0.25,
        zorder=2,
    )
    styled_planned.plot(
        ax=target_ax,
        color=styled_planned["cable_color"],
        linewidth=0.35,
        linestyle="--",
        alpha=0.85,
        zorder=2.2,
    )
    observed_tracks.plot(
        ax=target_ax,
        color="#FF4FA3",
        linewidth=1.35,
        alpha=0.92,
        zorder=3.4,
    )
    target_ax.scatter(
        landing_points.geometry.x,
        landing_points.geometry.y,
        marker="+",
        s=10,
        color="#FFFFFF",
        linewidths=0.25,
        alpha=0.80,
        zorder=4,
    )
    # No per-point coordinate labels: they hang the renderer at this map size
    target_ax.set_axis_off()


fig, ax = plt.subplots(figsize=(30, 16), dpi=120, facecolor="#090A18")
draw_map_layers(ax)
ax.set_xlim(-180, 180)
ax.set_ylim(-70, 85)
ax.set_title(
    "Submarine cables, WWF whale corridors, and public satellite tracks",
    color="#56E0C5",
    fontsize=24,
    loc="left",
    pad=18,
)

zoom_ax = fig.add_axes([0.075, 0.11, 0.27, 0.43], facecolor="#090A18")
draw_map_layers(zoom_ax)
zoom_ax.set_xlim(-80, -69)
zoom_ax.set_ylim(-46, 2)
zoom_ax.set_title(
    "Observed tracks: Chile to Galapagos", color="#FFB3CB", fontsize=11, pad=7
)
for spine in zoom_ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("#FFB3CB")
    spine.set_linewidth(0.8)

legend_handles = [
    Patch(
        facecolor="#F28CB1",
        edgecolor="#FFB3CB",
        alpha=0.45,
        label="WWF migration corridor",
    ),
    Line2D(
        [0], [0], color="#FF4FA3", linewidth=2.2,
        label="Observed ARGOS track (public subset)",
    ),
    Line2D(
        [0], [0], linestyle="none", marker="+", color="#FFFFFF", label="Landing point"
    ),
    Line2D(
        [0], [0], color="#B7F4E8", linewidth=1.2, linestyle="--",
        label="Planned cable route",
    ),
]
legend = ax.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.075),
    ncol=4,
    frameon=False,
    fontsize=8,
)
plt.setp(legend.get_texts(), color="#B7F4E8")
fig.text(
    0.012,
    0.012,
    "Sources: TeleGeography; WWF Blue Corridors; Hucke-Gaete et al. 2018 "
    "(DOI: 10.7717/peerj.4695/supp-1); GMRT v4.5",
    color="#6F7898",
    fontsize=8,
)
plt.tight_layout(rect=[0, 0.055, 1, 1])
gmrt_alt.close()
plt.show()
plt.close(fig)

print(f"Loaded {len(track_points):,} valid, unique ARGOS observations.")
print(f"Rendered {len(observed_tracks):,} observed tag trajectories.")


## 8. Retrieve descriptive metadata

The route GeoJSON does not contain dates or operational status. Those attributes are
available through one detail endpoint per cable. The following cell downloads the
metadata concurrently and saves the result as a local cache. Depending on the network,
the first run may take several minutes; later runs use the saved file.

Failed requests are recorded rather than silently discarded. A small number of missing
records would therefore remain visible as a limitation of the snapshot.


In [ ]:
def fetch_cable_detail(cable_id, attempts=3):
    url = DETAIL_URL.format(cable_id=cable_id)
    for attempt in range(attempts):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.json(), None
        except requests.RequestException as exc:
            if attempt == attempts - 1:
                return None, {"id": cable_id, "error": str(exc)}
            time.sleep(1.5 * (attempt + 1))


if METADATA_FILE.exists():
    metadata_payload = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
    print(f"Using cached file: {METADATA_FILE}")
else:
    index_response = requests.get(INDEX_URL, timeout=60)
    index_response.raise_for_status()
    cable_index = index_response.json()
    cable_ids = sorted({item["id"] for item in cable_index})

    records = []
    errors = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {
            executor.submit(fetch_cable_detail, cable_id): cable_id
            for cable_id in cable_ids
        }
        for future in as_completed(futures):
            record, error = future.result()
            if record is not None:
                records.append(record)
            if error is not None:
                errors.append(error)

    metadata_payload = {
        "source": INDEX_URL,
        "records": sorted(records, key=lambda item: item["id"]),
        "errors": errors,
    }
    METADATA_FILE.write_text(
        json.dumps(metadata_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved metadata cache: {METADATA_FILE}")

metadata = pd.DataFrame(metadata_payload["records"])
print(f"Metadata records: {len(metadata):,}")
print(f"Failed requests: {len(metadata_payload.get('errors', [])):,}")
metadata.head()

## 9. Clean and join the attributes

The published cable length is text such as `"45,000 km"`; it is converted into a
numeric value for summary statistics. Metadata are joined to the geometries using the
cable `id`. We keep in mind that a single cable can have more than one route feature,
so cable-level charts must use the metadata table rather than the geometry rows.


In [ ]:
def parse_length_km(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"[0-9,]+", str(value))
    if not match:
        return pd.NA
    return float(match.group(0).replace(",", ""))


metadata["length_km"] = metadata["length"].apply(parse_length_km).astype("Float64")
metadata["rfs_year"] = pd.to_numeric(metadata["rfs_year"], errors="coerce").astype(
    "Int64"
)
metadata["status"] = metadata["is_planned"].map({True: "Planned", False: "In service"})

cable_routes = cables.merge(
    metadata.drop(columns="landing_points", errors="ignore"),
    on="id",
    how="left",
    suffixes=("_geometry", "_metadata"),
)

print(f"Geometry features with an RFS year: {cable_routes['rfs_year'].notna().sum():,}")
metadata[["name", "rfs_year", "status", "length_km", "owners", "suppliers"]].head()

In [ ]:
metadata[["rfs_year", "length_km"]].describe().round(1)

## 10. Non-map visualization: cables by ready-for-service year

`rfs_year` records when a system was, or is expected to be, ready for service. Grouping
by year shows periods of infrastructure expansion. Planned cables are separated from
in-service cables because future dates are expectations rather than completed events.

Missing years are excluded from the chart but counted explicitly below.


In [ ]:
from matplotlib.ticker import MultipleLocator

yearly = (
    metadata.dropna(subset=["rfs_year", "status"])
    .groupby(["rfs_year", "status"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

x = range(len(yearly.index))
in_service = yearly.get("In service", pd.Series(0, index=yearly.index))
planned = yearly.get("Planned", pd.Series(0, index=yearly.index))

fig, ax = plt.subplots(figsize=(18, 10), facecolor="#090A18")
ax.set_facecolor("#090A18")
ax.bar(
    x,
    in_service,
    width=0.82,
    facecolor="none",
    edgecolor="#FF96FF",
    linewidth=1.5,
    label="In service",
)
ax.bar(
    x,
    planned,
    bottom=in_service,
    width=0.82,
    facecolor="none",
    edgecolor="#FFFFFF",
    linewidth=1.5,
    label="Planned",
)

ax.set_title("Submarine cable systems by ready-for-service year", color="#FFFFFF")
ax.set_xlabel("Ready-for-service year", color="#FFFFFF")
ax.set_ylabel("Number of cable systems", color="#FFFFFF")

legend = ax.legend(
    title="Status", facecolor="#090A18", edgecolor="#FFFFFF", labelcolor="#FFFFFF"
)
legend.get_title().set_color("#FFFFFF")

ax.set_xticks(list(x))
ax.set_xticklabels(
    yearly.index.astype(str), rotation=90, ha="center", va="top", color="#FFFFFF"
)
ax.tick_params(axis="y", colors="#FFFFFF")
ax.spines["bottom"].set_color("#FFFFFF")
ax.spines["left"].set_color("#FFFFFF")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.yaxis.set_major_locator(MultipleLocator(5))
ax.set_axisbelow(True)
ax.grid(axis="y", which="major", color="#FFFFFF", alpha=0.25)

plt.tight_layout()
plt.show()

print(f"Cable records without an RFS year: {metadata['rfs_year'].isna().sum():,}")

## 11. Cable systems by total length (ownership colours)

Each bar is one cable system ordered from longest to shortest. Colours match map
**3B2** (telco/consortium, Google, Meta, Microsoft, Amazon, multiple big tech).


In [ ]:
from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D

neon_palette = [
    "#F15BB5",  # magenta  -> telco / consortium
    "#00CFE8",  # cyan  -> Microsoft
    "#1C8CFF",  # blue  -> Google
    "#9B5DE5",  # violet -> multiple big tech
    "#FEE440",  # pink  -> Meta
    "#56E0C5",
    "#FEE440",
    "#FF8C42",  # orange -> Amazon / AWS
]

BIG_TECH_LABELS = {
    "Google": "Google",
    "Meta": "Meta",
    "Facebook": "Meta",
    "Microsoft": "Microsoft",
    "Amazon": "Amazon",
    "Amazon Web Services": "Amazon",
    "AWS": "Amazon",
}

ownership_colors = {
    "Telco / consortium": neon_palette[0],
    "Google": neon_palette[2],
    "Meta": neon_palette[4],
    "Microsoft": neon_palette[1],
    "Amazon": neon_palette[7],
    "Multiple big tech": neon_palette[3],
    "Unknown": "#6F7898",
}


def split_owners(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    text = str(raw).strip()
    if not text:
        return []
    return [part.strip() for part in text.split(",") if part.strip()]


def ownership_group(raw_owners):
    owners = split_owners(raw_owners)
    if not owners:
        return "Unknown"
    big = []
    for owner in owners:
        label = BIG_TECH_LABELS.get(owner)
        if label and label not in big:
            big.append(label)
    if not big:
        return "Telco / consortium"
    if len(big) == 1:
        return big[0]
    return "Multiple big tech"


def ownership_legend(ax_or_fig, on_figure=False):
    legend_order = [
        "Telco / consortium",
        "Google",
        "Meta",
        "Microsoft",
        "Amazon",
        "Multiple big tech",
    ]
    handles = [
        Line2D([0], [0], color=ownership_colors[name], linewidth=3, label=name)
        for name in legend_order
    ]
    if on_figure:
        legend = ax_or_fig.legend(
            handles=handles,
            title="Ownership group",
            loc="lower center",
            bbox_to_anchor=(0.5, -0.12),
            ncol=6,
            frameon=False,
            fontsize=8,
        )
    else:
        legend = ax_or_fig.legend(
            handles=handles,
            title="Ownership group",
            loc="upper right",
            frameon=False,
            fontsize=8,
        )
    plt.setp(legend.get_texts(), color="#FFFFFF")
    legend.get_title().set_color("#FFFFFF")
    return legend


length_base = metadata.dropna(subset=["length_km", "name"]).copy()
length_base["ownership_group"] = length_base["owners"].apply(ownership_group)
length_base["bar_color"] = length_base["ownership_group"].map(ownership_colors)

by_length = length_base.sort_values("length_km", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(18, 10), facecolor="#090A18")
ax.set_facecolor("#090A18")
ax.bar(
    range(len(by_length)),
    by_length["length_km"].astype(float),
    width=0.35,
    color=by_length["bar_color"].tolist(),
    edgecolor=by_length["bar_color"].tolist(),
    linewidth=0.35,
)

ax.set_title(
    "Submarine cable systems by length (longest to shortest)",
    color="#FFFFFF",
    loc="left",
)
ax.set_xlabel("Cable system", color="#FFFFFF")
ax.set_ylabel("Length (km)", color="#FFFFFF")

label_step = 1
tick_positions = list(range(0, len(by_length), label_step))
ax.set_xticks(tick_positions)
ax.set_xticklabels(
    by_length.loc[tick_positions, "name"].tolist(),
    rotation=90,
    ha="center",
    va="top",
    color="#FFFFFF",
    fontsize=3,
)
ax.tick_params(axis="y", colors="#FFFFFF")
ax.spines["bottom"].set_color("#FFFFFF")
ax.spines["left"].set_color("#FFFFFF")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_major_locator(MultipleLocator(2500))
ax.set_axisbelow(True)
ax.grid(axis="y", which="major", color="#FFFFFF", alpha=0.25)
ownership_legend(ax)

plt.tight_layout()
fig.savefig(
    "01-CableOwnership.png",
    dpi=600,
    facecolor="#090A18",
    edgecolor="none",
)
plt.show()
plt.close(fig)

print(f"Cables plotted: {len(by_length):,}")
print(by_length["ownership_group"].value_counts().to_string())

## 12. Cable systems by ready-for-service year (ownership colours)

Same systems as 6B, but grouped **chronologically**. Bars that share a year are
kept together, with a visible gap before the next year. The x-axis labels show
`year (n=…)`, and the printed table under the chart lists every cable included
in each year.


In [ ]:
from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D

neon_palette = [
    "#F15BB5",  # magenta  -> telco / consortium
    "#00CFE8",  # cyan  -> Microsoft
    "#1C8CFF",  # blue  -> Google
    "#9B5DE5",  # violet -> multiple big tech
    "#FEE440",  # pink  -> Meta
    "#56E0C5",
    "#FEE440",
    "#FF8C42",  # orange -> Amazon / AWS
]

BIG_TECH_LABELS = {
    "Google": "Google",
    "Meta": "Meta",
    "Facebook": "Meta",
    "Microsoft": "Microsoft",
    "Amazon": "Amazon",
    "Amazon Web Services": "Amazon",
    "AWS": "Amazon",
}

ownership_colors = {
    "Telco / consortium": neon_palette[0],
    "Google": neon_palette[2],
    "Meta": neon_palette[4],
    "Microsoft": neon_palette[1],
    "Amazon": neon_palette[7],
    "Multiple big tech": neon_palette[3],
    "Unknown": "#6F7898",
}


def split_owners(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    text = str(raw).strip()
    if not text:
        return []
    return [part.strip() for part in text.split(",") if part.strip()]


def ownership_group(raw_owners):
    owners = split_owners(raw_owners)
    if not owners:
        return "Unknown"
    big = []
    for owner in owners:
        label = BIG_TECH_LABELS.get(owner)
        if label and label not in big:
            big.append(label)
    if not big:
        return "Telco / consortium"
    if len(big) == 1:
        return big[0]
    return "Multiple big tech"


def ownership_legend(ax_or_fig, on_figure=False):
    legend_order = [
        "Telco / consortium",
        "Google",
        "Meta",
        "Microsoft",
        "Amazon",
        "Multiple big tech",
    ]
    handles = [
        Line2D([0], [0], color=ownership_colors[name], linewidth=3, label=name)
        for name in legend_order
    ]
    if on_figure:
        legend = ax_or_fig.legend(
            handles=handles,
            title="Ownership group",
            loc="lower center",
            bbox_to_anchor=(0.5, -0.12),
            ncol=6,
            frameon=False,
            fontsize=8,
        )
    else:
        legend = ax_or_fig.legend(
            handles=handles,
            title="Ownership group",
            loc="upper right",
            frameon=False,
            fontsize=8,
        )
    plt.setp(legend.get_texts(), color="#FFFFFF")
    legend.get_title().set_color("#FFFFFF")
    return legend


length_base = metadata.dropna(subset=["length_km", "name"]).copy()
length_base["ownership_group"] = length_base["owners"].apply(ownership_group)
length_base["bar_color"] = length_base["ownership_group"].map(ownership_colors)

by_year = (
    length_base.dropna(subset=["rfs_year"])
    .assign(rfs_year=lambda d: pd.to_numeric(d["rfs_year"], errors="coerce"))
    .dropna(subset=["rfs_year"])
    .sort_values(["rfs_year", "length_km"], ascending=[True, False])
    .copy()
)
by_year["rfs_year"] = by_year["rfs_year"].astype(int)

# Build x positions with a gap between successive years
year_gap = 1.0
x_positions = []
year_centers = {}
year_spans = {}
cursor = 0.0
for year, group in by_year.groupby("rfs_year", sort=True):
    start = cursor
    for _ in range(len(group)):
        x_positions.append(cursor)
        cursor += 1.0
    end = cursor - 1.0
    year_centers[year] = 0.5 * (start + end)
    year_spans[year] = (start - 0.5, end + 0.5)
    cursor += year_gap

by_year = by_year.reset_index(drop=True)
by_year["x"] = x_positions

fig, ax = plt.subplots(figsize=(20, 10), facecolor="#090A18")
ax.set_facecolor("#090A18")
ax.bar(
    by_year["x"],
    by_year["length_km"].astype(float),
    width=0.35,
    color=by_year["bar_color"].tolist(),
    edgecolor=by_year["bar_color"].tolist(),
    linewidth=0.35,
)

# Separators between years
ymax = float(by_year["length_km"].max())
for year, (left, right) in year_spans.items():
    ax.axvline(
        right + year_gap / 2.0, color="#FFFFFF", alpha=0.08, linewidth=0.8, zorder=0
    )

year_counts = by_year.groupby("rfs_year").size().to_dict()
tick_years = sorted(year_centers)
ax.set_xticks([year_centers[y] for y in tick_years])
ax.set_xticklabels(
    [f"{y}\n(n={year_counts[y]})" for y in tick_years],
    rotation=0,
    ha="center",
    color="#FFFFFF",
    fontsize=7,
)

ax.set_title(
    "Submarine cable systems grouped by ready-for-service year",
    color="#FFFFFF",
    loc="left",
)
ax.set_xlabel(
    "Ready-for-service year (each bar = one cable in that year)", color="#FFFFFF"
)
ax.set_ylabel("Length (km)", color="#FFFFFF")
ax.tick_params(axis="y", colors="#FFFFFF")
ax.spines["bottom"].set_color("#FFFFFF")
ax.spines["left"].set_color("#FFFFFF")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_major_locator(MultipleLocator(2500))
ax.set_axisbelow(True)
ax.grid(axis="y", which="major", color="#FFFFFF", alpha=0.25)
ownership_legend(ax, on_figure=False)

plt.tight_layout()
fig.savefig(
    "01-.png",
    dpi=600,
    facecolor="#090A18",
    edgecolor="none",
)
plt.show()
plt.close(fig)

# Explicit membership list: which cables belong to each year
membership_rows = []
for year, group in by_year.groupby("rfs_year", sort=True):
    cable_bits = [
        f"{name} ({length:.0f} km, {own})"
        for name, length, own in zip(
            group["name"], group["length_km"], group["ownership_group"]
        )
    ]
    membership_rows.append(
        {
            "year": int(year),
            "n": len(group),
            "cables (name, length, ownership)": "; ".join(cable_bits),
        }
    )
cables_by_year = pd.DataFrame(membership_rows)

print(
    f"Cables plotted chronologically: {len(by_year):,} "
    f"across {by_year['rfs_year'].nunique()} years"
)
print()
print("Cables included in each ready-for-service year:")
pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_rows", 200)
display(cables_by_year)